In [ ]:
"""
Polar Surface Water and Atlantic Water Comparison Figure
=========================================================
6-panel figure comparing PSW (Fram Strait) and AW (Eastern Boundary) properties and fluxes.

Layout (3 rows × 2 cols):
  Col 1 (PSW)              Col 2 (AW)
  a) Net Transport         b) Net Transport
  c) Temperature           d) Temperature
  e) Salinity              f) Salinity

PSW: S < 34.5 psu AND ρ < 1027.7 kg/m³ at Fram Strait (79°N, -20°W to 12°E)
AW: Geographic definition (lon > 5°E, depth < 550m) at Eastern Boundary

Style: Matches Figure 2 (split trends on monthly data with bootstrap CI)
Output: Single 6-panel figure
Version: 1.2.0
Last Modified: 22-06-2026
Author: Chris Barrell
"""
import xarray as xr
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as dates
from scipy import stats as scipy_stats
import pymannkendall as mk
import os
import sys
from pathlib import Path
from datetime import datetime

# Setup logging
sys.path.append('..')
from utils.logger import (setup_logger, log_data_loading, log_processing_step,
                          log_output_file, log_completion, log_error)

logger = setup_logger('figure_04_PSW_and_AW', config_path='../config.yaml')
start_time = datetime.now()

try:
    # ========================================================================
    # CONFIGURATION
    # ========================================================================
    
    logger.info("Loading configuration...")
    
    # Data paths
    GLORYS_PATH = '../../glorys12_with_density/glorys12_*_Greenland_Sea_with_density.nc'
    
    # Create output directories
    OUTPUT_DIR = Path('./outputs/figures')
    PROCESSED_DIR = Path('./outputs/processed_data/figure_04')
    OUTPUT_METHODS_DIR = Path('./outputs/methods')
    
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
    OUTPUT_METHODS_DIR.mkdir(parents=True, exist_ok=True)
    
    # Greenland Sea boundary corners
    GREENLAND_SEA_COORDS = [
        (-22, 71),    # SW
        (-8.5, 71),   # SE
        (12, 79),     # NE
        (-21, 79)     # NW
    ]
    
    # PSW definition
    PSW_S_THRESHOLD = 34.5      # psu
    PSW_RHO_THRESHOLD = 1027.7  # kg/m³
    
    # Fram Strait location
    FRAM_STRAIT_LAT = 79.0
    FRAM_STRAIT_LON_MIN = -20.0
    FRAM_STRAIT_LON_MAX = 12.0
    
    # AW geographic definition
    AW_LON_THRESHOLD = 5.0   # °E
    AW_MAX_DEPTH = 550.0     # m
    
    # Processing options
    REPROCESS = False
    
    # Analysis configuration
    SPLIT_YEAR = 2015
    SHOW_CONFIDENCE_INTERVALS = True
    CONFIDENCE_LEVEL = 0.95
    N_BOOTSTRAP = 1000

    # Rolling mean configuration
    SHOW_ROLLING_MEAN = True   # Set False to omit smooth overlay
    ROLLING_WIN = 12           # months
    ROLLING_HALF = ROLLING_WIN // 2   # 6 months — excluded at each end
    
    # Colors
    COLOR_PSW = '#2E86AB'  # Blue
    COLOR_AW = '#F77F00'   # Orange
    
    logger.info("Configuration loaded successfully")
    logger.info(f"  PSW definition: S < {PSW_S_THRESHOLD} psu, ρ < {PSW_RHO_THRESHOLD} kg/m³")
    logger.info(f"  AW definition: lon > {AW_LON_THRESHOLD}°E, depth < {AW_MAX_DEPTH} m")
    logger.info(f"  Split year: {SPLIT_YEAR}")
    logger.info(f"  Bootstrap iterations: {N_BOOTSTRAP}")
    logger.info(f"  Rolling mean: {'enabled' if SHOW_ROLLING_MEAN else 'disabled'} "
                f"(window={ROLLING_WIN} months, {ROLLING_HALF} months masked at each end)")
    
    # ========================================================================
    # UTILITY FUNCTIONS
    # ========================================================================
    
    def haversine_distance(lat1, lon1, lat2, lon2):
        """Calculate great circle distance in km."""
        R = 6371
        lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
        dlat = lat2 - lat1
        dlon = lon2 - lon1
        a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
        c = 2 * np.arcsin(np.sqrt(a))
        return R * c
    
    # ========================================================================
    # STATISTICAL FUNCTIONS
    # ========================================================================
    
    def mann_kendall_test(x, y):
        """
        Yue & Wang (2004) modified Mann-Kendall test accounting for autocorrelation.

        Uses pymannkendall.yue_wang_modification_test, which corrects the variance
        of the MK statistic using serial correlation coefficients at all lags.
        Only the significance test is affected; trend magnitude is unchanged.

        References
        ----------
        Yue, S., & Wang, C. (2004). The Mann-Kendall test modified by effective
        sample size to detect trend in serially correlated hydrological series.
        Water Resources Management, 18(3), 201-218.
        https://doi.org/10.1023/B:WARM.0000043140.61082.60
        """
        y = np.asarray(y)
        valid = ~np.isnan(y)
        y = y[valid]

        result = mk.yue_wang_modification_test(y)

        if result.p < 0.001:
            sig = "***"
        elif result.p < 0.01:
            sig = "**"
        elif result.p < 0.05:
            sig = "*"
        else:
            sig = ""

        return {
            'tau': result.Tau,
            'p_value': result.p,
            'significance': sig,
            'z_statistic': result.z
        }
    
    def get_trend(x, y):
        """
        Calculate linear trend with statistics using ordinary least squares.
        """
        mask = ~np.isnan(y)
        x_clean = x[mask]
        y_clean = y[mask]
        
        if len(x_clean) < 2:
            return None, None, None
        
        p = np.polyfit(x_clean, y_clean, 1)
        trend = np.polyval(p, x)
        
        slope, intercept, r_value, p_value, std_err = scipy_stats.linregress(x_clean, y_clean)
        
        stats_dict = {
            'slope': slope,
            'intercept': intercept,
            'r_value': r_value,
            'r_squared': r_value**2,
            'p_value': p_value,
            'std_err': std_err
        }
        
        return p, trend, stats_dict
    
    def bootstrap_trend_ci(x, y, n_bootstrap=1000, confidence_level=0.95):
        """
        Bootstrap confidence intervals for trend line.

        References
        ----------
        Efron, B., & Tibshirani, R. J. (1994). An Introduction to the Bootstrap.
        CRC press.
        """
        mask = ~np.isnan(y)
        x_clean = x[mask]
        y_clean = y[mask]
        
        n = len(x_clean)
        trends = []
        
        for _ in range(n_bootstrap):
            idx = np.random.choice(n, size=n, replace=True)
            x_boot = x_clean[idx]
            y_boot = y_clean[idx]
            
            p_boot = np.polyfit(x_boot, y_boot, 1)
            trend_boot = np.polyval(p_boot, x)
            trends.append(trend_boot)
        
        trends = np.array(trends)
        
        alpha = 1 - confidence_level
        lower = np.percentile(trends, (alpha/2) * 100, axis=0)
        upper = np.percentile(trends, (1 - alpha/2) * 100, axis=0)
        
        return lower, upper

    def compute_rolling_mean(time_pd, data_arr):
        """
        Compute 12-month centred rolling mean with boundary masking.

        The first and last ROLLING_HALF months are set to NaN so the smooth
        line is not plotted where the centred window is underpopulated.

        Parameters
        ----------
        time_pd : pd.DatetimeIndex
        data_arr : np.ndarray

        Returns
        -------
        np.ndarray
            Smooth values, NaN at both ends (length matches input).
        """
        s = pd.Series(data_arr, index=time_pd)
        smooth = s.rolling(ROLLING_WIN, center=True, min_periods=6).mean().values
        smooth[:ROLLING_HALF] = np.nan
        smooth[-ROLLING_HALF:] = np.nan
        return smooth

    # ========================================================================
    # PSW CALCULATION FUNCTIONS
    # ========================================================================
    
    def calculate_psw_flux(ds, s_thresh, rho_thresh):
        """
        Calculate Polar Surface Water flux at Fram Strait.

        PSW Definition: S < s_thresh AND ρ < rho_thresh
        Sign Convention: Positive = southward = into Greenland Sea
        Units: Sv
        """
        log_processing_step(logger, "Calculating PSW flux at Fram Strait")
        
        FS = ds.sel(latitude=FRAM_STRAIT_LAT, method='nearest')
        FS = FS.sel(longitude=slice(FRAM_STRAIT_LON_MIN, FRAM_STRAIT_LON_MAX))
        
        logger.info(f"  Latitude: {float(FS.latitude.values):.2f}°N")
        logger.info(f"  Longitude range: {FRAM_STRAIT_LON_MIN}° to {FRAM_STRAIT_LON_MAX}°E")
        
        psw_mask = ((FS.so < s_thresh) & (FS.absolute_density < rho_thresh)).astype(float)
        
        lons = FS.longitude.values
        dx_list = []
        for i in range(len(lons) - 1):
            dist = haversine_distance(FRAM_STRAIT_LAT, lons[i], 
                                      FRAM_STRAIT_LAT, lons[i+1])
            dx_list.append(dist * 1000)
        dx_m = np.mean(dx_list)
        
        logger.info(f"  Mean dx: {dx_m:.1f} m")
        
        FS_sliced = FS.isel(longitude=slice(0, -1))
        psw_mask_sliced = psw_mask.isel(longitude=slice(0, -1))
        
        flux = -1 * (FS_sliced.vo * psw_mask_sliced * 
                     FS_sliced.delta_z * dx_m).sum(dim=['depth', 'longitude'])
        
        flux_sv = flux / 1e6
        logger.info(f"  Mean flux: {float(flux_sv.mean()):+.3f} Sv")
        
        return flux_sv
    
    def calculate_psw_properties(ds, s_thresh, rho_thresh):
        """
        Calculate mean temperature and salinity of PSW at Fram Strait.

        Method: Weighted mean where PSW exists.
        """
        log_processing_step(logger, "Calculating PSW properties at Fram Strait")
        
        FS = ds.sel(latitude=FRAM_STRAIT_LAT, method='nearest')
        FS = FS.sel(longitude=slice(FRAM_STRAIT_LON_MIN, FRAM_STRAIT_LON_MAX))
        
        psw_mask = ((FS.so < s_thresh) & (FS.absolute_density < rho_thresh)).astype(float)
        
        temp_mean = (FS.thetao * psw_mask).sum(dim=['depth', 'longitude']) / psw_mask.sum(dim=['depth', 'longitude'])
        sal_mean = (FS.so * psw_mask).sum(dim=['depth', 'longitude']) / psw_mask.sum(dim=['depth', 'longitude'])
        
        logger.info(f"  Mean T: {float(temp_mean.mean()):.2f}°C")
        logger.info(f"  Mean S: {float(sal_mean.mean()):.2f} psu")
        
        return temp_mean, sal_mean
    
    # ========================================================================
    # AW CALCULATION FUNCTIONS
    # ========================================================================
    
    def calculate_aw_flux_geographic(ds, lon_threshold, max_depth):
        """
        Calculate Atlantic Water flux at eastern boundary (geographic definition).

        AW Definition: lon > lon_threshold AND depth < max_depth
        Sign Convention: Positive = into Greenland Sea (westward)
        Units: Sv
        """
        log_processing_step(logger, "Calculating AW flux at eastern boundary (geographic definition)")
        
        p1 = GREENLAND_SEA_COORDS[1]  # (-8.5, 71)
        p2 = GREENLAND_SEA_COORDS[2]  # (12, 79)
        
        n_points = 100
        lons_line = np.linspace(p1[0], p2[0], n_points)
        lats_line = np.linspace(p1[1], p2[1], n_points)
        
        mask_lon = lons_line > lon_threshold
        lons_filtered = lons_line[mask_lon]
        lats_filtered = lats_line[mask_lon]
        
        logger.info(f"  Total interpolation points: {n_points}")
        logger.info(f"  Points where lon > {lon_threshold}°E: {len(lons_filtered)}")
        logger.info(f"  Longitude range: {lons_filtered.min():.2f}°E to {lons_filtered.max():.2f}°E")
        
        dx_line = p2[0] - p1[0]
        dy_line = p2[1] - p1[1]
        
        nx = dy_line
        ny = dx_line
        norm = np.sqrt(nx**2 + ny**2)
        nx /= norm
        ny /= norm
        
        logger.info(f"  Normal vector: ({nx:.3f}, {ny:.3f})")
        
        flux_segments = []
        
        for i in range(len(lons_filtered)):
            point_data = ds.interp(longitude=lons_filtered[i], latitude=lats_filtered[i], method='linear')
            point_data_shallow = point_data.sel(depth=point_data.depth < max_depth)
            
            v_normal = nx * point_data_shallow.uo + ny * point_data_shallow.vo
            
            if i < len(lons_filtered) - 1:
                ds_m = haversine_distance(lats_filtered[i], lons_filtered[i],
                                         lats_filtered[i+1], lons_filtered[i+1]) * 1000
            else:
                ds_m = 0
            
            flux_segment = (v_normal * point_data_shallow.delta_z * ds_m).sum(dim='depth')
            flux_segments.append(flux_segment)
        
        flux = sum(flux_segments)
        flux_sv = flux / 1e6
        
        logger.info(f"  Mean flux: {float(flux_sv.mean()):.3f} Sv")
        
        return flux_sv
    
    def calculate_aw_properties_geographic(ds, lon_threshold, max_depth):
        """
        Calculate mean T and S at eastern boundary (geographic definition).

        Method: Simple mean over filtered points and depth.
        """
        log_processing_step(logger, "Calculating AW properties at eastern boundary (geographic definition)")
        
        p1 = GREENLAND_SEA_COORDS[1]  # (-8.5, 71)
        p2 = GREENLAND_SEA_COORDS[2]  # (12, 79)
        
        n_points = 100
        lons_line = np.linspace(p1[0], p2[0], n_points)
        lats_line = np.linspace(p1[1], p2[1], n_points)
        
        mask_lon = lons_line > lon_threshold
        lons_filtered = lons_line[mask_lon]
        lats_filtered = lats_line[mask_lon]
        
        temp_segments = []
        sal_segments = []
        
        for lon, lat in zip(lons_filtered, lats_filtered):
            point_data = ds.interp(longitude=lon, latitude=lat, method='linear')
            point_data_shallow = point_data.sel(depth=point_data.depth < max_depth)
            
            temp_segments.append(point_data_shallow.thetao)
            sal_segments.append(point_data_shallow.so)
        
        temp_all = xr.concat(temp_segments, dim='point')
        sal_all = xr.concat(sal_segments, dim='point')
        
        temp_mean = temp_all.mean(dim=['point', 'depth'])
        sal_mean = sal_all.mean(dim=['point', 'depth'])
        
        logger.info(f"  Mean T: {float(temp_mean.mean()):.2f}°C")
        logger.info(f"  Mean S: {float(sal_mean.mean()):.2f} psu")
        
        return temp_mean, sal_mean
    
    # ========================================================================
    # PLOTTING FUNCTION
    # ========================================================================
    
    def plot_panel(ax, time, data, color, ylabel, title, show_xlabel=False):
        """
        Plot single panel with split trends, bootstrap CIs, and optional
        12-month rolling mean.

        Rolling mean is computed with a centred 12-month window; the first
        and last ROLLING_HALF months are masked to avoid edge artefacts from
        underpopulated windows.

        Parameters
        ----------
        ax : matplotlib.axes.Axes
        time : pandas.DatetimeIndex
        data : np.ndarray
        color : str
        ylabel : str
        title : str
        show_xlabel : bool
        """
        
        # Split trend analysis
        mask_1 = time.year < SPLIT_YEAR
        mask_2 = time.year >= SPLIT_YEAR
        
        x_1 = time[mask_1]
        y_1 = data[mask_1]
        x_num_1 = dates.date2num(x_1)
        
        x_2 = time[mask_2]
        y_2 = data[mask_2]
        x_num_2 = dates.date2num(x_2)
        
        # Calculate trends
        p_1, trend_1, stats_1 = get_trend(x_num_1, y_1)
        mk_1 = mann_kendall_test(x_num_1, y_1)
        
        p_2, trend_2, stats_2 = get_trend(x_num_2, y_2)
        mk_2 = mann_kendall_test(x_num_2, y_2)
        
        # Bootstrap CIs
        if SHOW_CONFIDENCE_INTERVALS:
            ci_1_lower, ci_1_upper = bootstrap_trend_ci(x_num_1, y_1, N_BOOTSTRAP, CONFIDENCE_LEVEL)
            ci_2_lower, ci_2_upper = bootstrap_trend_ci(x_num_2, y_2, N_BOOTSTRAP, CONFIDENCE_LEVEL)
        
        # --- Plot monthly data ---
        ax.plot(time, data, '-', color=color, linewidth=1, alpha=0.4, zorder=2)

        # --- Plot 12-month rolling mean ---
        if SHOW_ROLLING_MEAN:
            smooth = compute_rolling_mean(time, data)
            # Plot as a solid darker line over the monthly data
            ax.plot(time, smooth, '-', color=color, linewidth=2.0, alpha=0.9, zorder=3,
                    label='12-month mean')
        
        # --- Bootstrap CI shading ---
        if SHOW_CONFIDENCE_INTERVALS:
            ax.fill_between(x_1, ci_1_lower, ci_1_upper, color=color, alpha=0.15, zorder=1)
            ax.fill_between(x_2, ci_2_lower, ci_2_upper, color=color, alpha=0.15, zorder=1)
        
        # --- Trend lines ---
        ax.plot(x_1, trend_1, '--', color=color, linewidth=2.5, alpha=0.9, zorder=4)
        ax.plot(x_2, trend_2, '--', color=color, linewidth=2.5, alpha=0.9, zorder=4)
        
        # --- Split year marker ---
        ax.axvline(x=np.datetime64(f'{SPLIT_YEAR}-01-01'), 
                   color='gray', linestyle=':', linewidth=1.5, alpha=0.6, zorder=0)
        
        # --- Zero line for transport panels ---
        if 'Transport' in ylabel:
            ax.axhline(y=0, color='gray', linestyle='-', linewidth=0.8, alpha=0.4, zorder=0)
        
        # --- Stats box ---
        slope_1_dec = stats_1['slope'] * 3652.5
        slope_2_dec = stats_2['slope'] * 3652.5
        
        if 'Sv' in ylabel:
            unit_str = 'Sv dec\u207b\u00b9'
        elif 'Temperature' in ylabel or '°C' in ylabel:
            unit_str = '\u00b0C dec\u207b\u00b9'
        else:
            unit_str = 'psu dec\u207b\u00b9'
        
        stats_text = (
        # TODO: update Period 1 display label to 1993-{SPLIT_YEAR-1} when P1 label convention is standardised across all figures
            f"1993\u2013{SPLIT_YEAR}: {slope_1_dec:+.4f} {unit_str}{mk_1['significance']}\n"
            f"{SPLIT_YEAR}\u20132025: {slope_2_dec:+.4f} {unit_str}{mk_2['significance']}"
        )
        
        ax.text(0.02, 0.98, stats_text, transform=ax.transAxes, fontsize=9,
                verticalalignment='top',
                bbox=dict(boxstyle='round', facecolor='white', alpha=0.85),
                family='monospace')
        
        # --- Labels and styling ---
        ax.set_ylabel(ylabel, fontsize=11)
        if show_xlabel:
            ax.set_xlabel('Year', fontsize=11)
        ax.grid(True, alpha=0.3, linewidth=0.5)
        ax.set_title(title, fontsize=11, loc='left', fontweight='normal')
    
    # ========================================================================
    # METHODOLOGY DOCUMENTATION
    # ========================================================================
    
    def write_methodology(methods_dir):
        """Write comprehensive methodology documentation."""
        
        methodology_file = methods_dir / 'figure_04_PSW_and_AW_methods.md'
        
        with open(methodology_file, 'w') as f:
            f.write("="*80 + "\n")
            f.write("POLAR SURFACE WATER AND ATLANTIC WATER COMPARISON FIGURE\n")
            f.write("METHODOLOGY DOCUMENTATION\n")
            f.write("="*80 + "\n")
            f.write(f"\nGenerated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
            f.write(f"Script: figure_04_PSW_and_AW.ipynb\n")
            
            f.write("\n" + "="*80 + "\n")
            f.write("1. DATA SOURCES\n")
            f.write("="*80 + "\n")
            f.write("\nGLORYS12 Ocean Reanalysis (monthly, 1993-2025)\n")
            f.write("  Variables used:\n")
            f.write("    - thetao: Potential temperature (°C)\n")
            f.write("    - so: Salinity (psu)\n")
            f.write("    - absolute_density: In-situ density (kg/m³)\n")
            f.write("    - vo: Meridional velocity (m/s)\n")
            f.write("    - uo: Zonal velocity (m/s)\n")
            f.write("    - delta_z: Layer thickness (m)\n")
            f.write("  Horizontal resolution: ~1/12° (~9 km)\n")
            f.write("  Vertical levels: 50 depth levels\n")
            
            f.write("\n" + "="*80 + "\n")
            f.write("2. WATER MASS DEFINITIONS\n")
            f.write("="*80 + "\n")
            f.write("\n2.1 Polar Surface Water (PSW)\n")
            f.write("  Definition: S < 34.5 psu AND ρ < 1027.7 kg/m³\n")
            f.write("  Location: Fram Strait (79°N, -20°W to 12°E)\n")
            f.write("  Depth range: All depths\n")
            f.write("  Rationale: Captures fresh, low-density Arctic outflow\n")
            f.write("  Note: absolute_density is in-situ density (kg/m³) computed via\n")
            f.write("        TEOS-10 from GLORYS12 temperature, salinity, and pressure;\n")
            f.write("        it is a derived variable in the pre-processed GLORYS12\n")
            f.write("        collocated files, not a native Copernicus download variable.\n")
            
            
            f.write("\n2.2 Atlantic Water (AW)\n")
            f.write("  Definition: GEOGRAPHIC (lon > 5°E AND depth < 550 m)\n")
            f.write("  Location: Eastern Greenland Sea boundary\n")
            f.write("              Diagonal from (-8.5°, 71°N) to (12°E, 79°N)\n")
            f.write("  Rationale: Captures main AW inflow corridor without\n")
            f.write("             temperature/salinity thresholds that may\n")
            f.write("             vary with climate change\n")
            
            f.write("\n" + "="*80 + "\n")
            f.write("3. NET TRANSPORT CALCULATIONS\n")
            f.write("="*80 + "\n")
            f.write("\n3.1 PSW Net Transport at Fram Strait\n")
            f.write("  Formula: Transport = -vo × PSW_mask × dz × dx\n")
            f.write("           where PSW_mask = 1 where PSW criteria met, 0 elsewhere\n")
            f.write("  dx calculation: Haversine distance between longitude points\n")
            f.write("  Sign convention: Positive = southward (into Greenland Sea)\n")
            f.write("  Units: Sv (Sverdrups = 10^6 m³/s)\n")
            f.write("  Integration: Sum over depth and longitude dimensions\n")
            
            f.write("\n3.2 AW Net Transport at Eastern Boundary\n")
            f.write("  Method: Normal velocity component through diagonal boundary\n")
            f.write("  Steps:\n")
            f.write("    1. Define 100 interpolation points along diagonal boundary\n")
            f.write("    2. Filter to points where lon > 5°E\n")
            f.write("    3. Select depths < 550 m\n")
            f.write("    4. Calculate normal vector (perpendicular to boundary)\n")
            f.write("       Normal = (dy_line, dx_line) / ||(dy_line, dx_line)||\n")
            f.write("    5. Calculate normal velocity: v_normal = nx × u + ny × v\n")
            f.write("    6. Calculate transport through each segment:\n")
            f.write("       Transport_segment = v_normal × dz × ds\n")
            f.write("       where ds = haversine distance between points\n")
            f.write("    7. Sum all segments\n")
            f.write("  Sign convention: Positive = into Greenland Sea (westward)\n")
            f.write("  Units: Sv (Sverdrups = 10^6 m³/s)\n")
            
            f.write("\n" + "="*80 + "\n")
            f.write("4. PROPERTY CALCULATIONS\n")
            f.write("="*80 + "\n")
            f.write("\n4.1 PSW Properties\n")
            f.write("  Method: Weighted mean where PSW exists\n")
            f.write("  Formula: Mean_T = sum(T × PSW_mask) / sum(PSW_mask)\n")
            f.write("           Mean_S = sum(S × PSW_mask) / sum(PSW_mask)\n")
            f.write("  Integration: Over depth and longitude at Fram Strait\n")
            
            f.write("\n4.2 AW Properties\n")
            f.write("  Method: Simple mean in geographic region\n")
            f.write("  Formula: Mean_T = mean(T) over filtered points and depths\n")
            f.write("           Mean_S = mean(S) over filtered points and depths\n")
            f.write("  Integration: Over interpolation points (lon > 5°E) and depths < 550 m\n")
            
            f.write("\n" + "="*80 + "\n")
            f.write("5. STATISTICAL ANALYSIS\n")
            f.write("="*80 + "\n")
            f.write("\n5.1 Linear Trend Analysis\n")
            f.write("  Method: Piecewise linear regression with fixed breakpoint\n")
            f.write(f"  Breakpoint year: {SPLIT_YEAR} (pre-specified on physical grounds)\n")
            # TODO: update Period 1 label to 1993-2014 when P1 label convention is standardised
            f.write("  Period 1: 1993-2015\n")
            f.write("  Period 2: 2015-2025\n")
            f.write("  Fitting: Ordinary least squares (scipy.stats.linregress)\n")
            f.write("  Units: Original units per decade (slope × 3652.5)\n")
            
            f.write("\n5.2 Significance Testing\n")
            f.write("  Test: Modified Mann-Kendall trend test (Yue & Wang, 2004)\n")
            f.write("  Correction: Variance corrected using serial correlation coefficients\n")
            f.write("              at all lags to account for autocorrelation in monthly\n")
            f.write("              timeseries\n")
            f.write("  Null hypothesis: No monotonic trend\n")
            f.write("  Significance levels:\n")
            f.write("    * p < 0.05\n")
            f.write("    ** p < 0.01\n")
            f.write("    *** p < 0.001\n")
            f.write("  References:\n")
            f.write("    Yue, S., & Wang, C. (2004). Water Resources Management, 18(3), 201-218.\n")
            f.write("    https://doi.org/10.1023/B:WARM.0000043140.61082.60\n")
            
            f.write("\n5.3 Uncertainty Quantification\n")
            f.write("  Method: Bootstrap resampling\n")
            f.write(f"  Iterations: {N_BOOTSTRAP}\n")
            f.write(f"  Confidence level: {CONFIDENCE_LEVEL*100:.0f}%\n")
            f.write("  Procedure:\n")
            f.write("    1. Randomly resample data with replacement\n")
            f.write("    2. Calculate trend for bootstrap sample\n")
            f.write("    3. Repeat 1000 times\n")
            f.write("    4. Extract 2.5th and 97.5th percentiles\n")
            f.write("  Reference:\n")
            f.write("    Efron & Tibshirani (1994). An Introduction to the Bootstrap.\n")
            
            f.write("\n5.4 Rolling Mean Smoothing\n")
            f.write(f"  Window: {ROLLING_WIN} months, centred\n")
            f.write(f"  Minimum periods: 6\n")
            f.write(f"  Boundary masking: First and last {ROLLING_HALF} months set to NaN\n")
            f.write("  Rationale: Centred windows are underpopulated near record ends,\n")
            f.write("             causing spurious edge curvature; masking prevents this.\n")
            f.write("             Monthly data continues to record endpoints.\n")
            
            f.write("\n" + "="*80 + "\n")
            f.write("6. VISUALIZATION\n")
            f.write("="*80 + "\n")
            f.write("\nFigure Layout: 6 panels (3 rows × 2 columns)\n")
            f.write("  Column 1: Polar Surface Water (PSW)\n")
            f.write("  Column 2: Atlantic Water (AW)\n")
            f.write("  Row 1: Net transport (Sv)\n")
            f.write("  Row 2: Temperature (°C)\n")
            f.write("  Row 3: Salinity (psu)\n")
            f.write("\nPanel Elements:\n")
            f.write("  - Monthly data: solid line, α=0.4\n")
            f.write(f"  - {'12-month rolling mean: solid line, lw=2.0, α=0.9' if SHOW_ROLLING_MEAN else 'Rolling mean: disabled'}\n")
            f.write("  - Trend lines: dashed, linewidth=2.5, α=0.9\n")
            f.write("  - Bootstrap CI: shaded region, α=0.15\n")
            f.write("  - Breakpoint: gray dotted vertical line\n")
            f.write("  - Zero line: gray horizontal (transport panels only)\n")
            f.write("  - Grid: α=0.3\n")
            f.write("\nColors:\n")
            f.write(f"  PSW: {COLOR_PSW} (blue)\n")
            f.write(f"  AW: {COLOR_AW} (orange)\n")
            
            f.write("\n" + "="*80 + "\n")
            f.write("7. REFERENCES\n")
            f.write("="*80 + "\n")
            f.write("\nGLORYS12 Reanalysis:\n")
            f.write("  Lellouche, J.-M., et al. (2021). The Copernicus Global 1/12°\n")
            f.write("  Oceanic and Sea Ice GLORYS12 Reanalysis. Frontiers in Earth\n")
            f.write("  Science, 9, 698876.\n")
            f.write("\nStatistical Methods:\n")
            f.write("  Yue, S., & Wang, C. (2004). The Mann-Kendall test modified by\n")
            f.write("  effective sample size to detect trend in serially correlated\n")
            f.write("  hydrological series. Water Resources Management, 18(3), 201-218.\n")
            f.write("  https://doi.org/10.1023/B:WARM.0000043140.61082.60\n")
            f.write("  Efron, B., & Tibshirani, R. J. (1994). An Introduction to the\n")
            f.write("  Bootstrap. CRC Press.\n")
            
            f.write("\n" + "="*80 + "\n")
            f.write("END OF METHODOLOGY DOCUMENTATION\n")
            f.write("="*80 + "\n")
        
        log_output_file(logger, methodology_file, "Methodology documentation")
        return methodology_file
    
    # ========================================================================
    # MAIN ANALYSIS
    # ========================================================================
    
    logger.info("")
    logger.info("="*70)
    logger.info("STARTING PSW AND AW COMPARISON ANALYSIS")
    logger.info("="*70)
    
    # Define filenames
    psw_flux_csv = PROCESSED_DIR / f'psw_flux_S{PSW_S_THRESHOLD}_rho{PSW_RHO_THRESHOLD}.csv'
    psw_props_csv = PROCESSED_DIR / f'psw_properties_S{PSW_S_THRESHOLD}_rho{PSW_RHO_THRESHOLD}.csv'
    aw_flux_csv = PROCESSED_DIR / f'aw_flux_lon{AW_LON_THRESHOLD}E_depth{int(AW_MAX_DEPTH)}m.csv'
    aw_props_csv = PROCESSED_DIR / f'aw_properties_lon{AW_LON_THRESHOLD}E_depth{int(AW_MAX_DEPTH)}m.csv'
    
    # Check if we need to reprocess
    if REPROCESS or not all([f.exists() for f in [psw_flux_csv, psw_props_csv, aw_flux_csv, aw_props_csv]]):
        log_data_loading(logger, "GLORYS12", GLORYS_PATH)
        
        ds = xr.open_mfdataset(GLORYS_PATH, parallel=False)
        
        logger.info(f"Time range: {pd.to_datetime(ds.time.values[0]).strftime('%Y-%m')} to "
              f"{pd.to_datetime(ds.time.values[-1]).strftime('%Y-%m')}")
        logger.info(f"Total months: {len(ds.time)}")
        
        logger.info("")
        logger.info("="*70)
        logger.info("CALCULATING PSW DATA")
        logger.info("="*70)
        
        psw_flux = calculate_psw_flux(ds, PSW_S_THRESHOLD, PSW_RHO_THRESHOLD)
        psw_temp, psw_sal = calculate_psw_properties(ds, PSW_S_THRESHOLD, PSW_RHO_THRESHOLD)
        
        df_psw_flux = pd.DataFrame({'time': ds.time.values, 'flux_sv': psw_flux.values})
        df_psw_props = pd.DataFrame({'time': ds.time.values, 'temp': psw_temp.values, 'sal': psw_sal.values})
        
        df_psw_flux.to_csv(psw_flux_csv, index=False)
        df_psw_props.to_csv(psw_props_csv, index=False)
        log_output_file(logger, psw_flux_csv, "PSW flux data")
        log_output_file(logger, psw_props_csv, "PSW properties data")
        
        logger.info("")
        logger.info("="*70)
        logger.info("CALCULATING AW DATA")
        logger.info("="*70)
        
        aw_flux = calculate_aw_flux_geographic(ds, AW_LON_THRESHOLD, AW_MAX_DEPTH)
        aw_temp, aw_sal = calculate_aw_properties_geographic(ds, AW_LON_THRESHOLD, AW_MAX_DEPTH)
        
        df_aw_flux = pd.DataFrame({'time': ds.time.values, 'flux_sv': aw_flux.values})
        df_aw_props = pd.DataFrame({'time': ds.time.values, 'temp': aw_temp.values, 'sal': aw_sal.values})
        
        df_aw_flux.to_csv(aw_flux_csv, index=False)
        df_aw_props.to_csv(aw_props_csv, index=False)
        log_output_file(logger, aw_flux_csv, "AW flux data")
        log_output_file(logger, aw_props_csv, "AW properties data")
        
    else:
        logger.info("Loading processed data from CSV files...")
        logger.info("  (Set REPROCESS=True to recalculate)")
        
        df_psw_flux = pd.read_csv(psw_flux_csv)
        df_psw_props = pd.read_csv(psw_props_csv)
        df_aw_flux = pd.read_csv(aw_flux_csv)
        df_aw_props = pd.read_csv(aw_props_csv)
        
        for df in [df_psw_flux, df_psw_props, df_aw_flux, df_aw_props]:
            df['time'] = pd.to_datetime(df['time'])
        
        log_data_loading(logger, "PSW flux", psw_flux_csv)
        log_data_loading(logger, "PSW properties", psw_props_csv)
        log_data_loading(logger, "AW flux", aw_flux_csv)
        log_data_loading(logger, "AW properties", aw_props_csv)
    
    # Convert to datetime
    time = pd.to_datetime(df_psw_flux['time'].values)
    
    # ========================================================================
    # CREATE 6-PANEL FIGURE (3 rows × 2 cols)
    # ========================================================================
    
    log_processing_step(logger, "Creating 6-panel comparison figure")
    
    fig, axes = plt.subplots(3, 2, figsize=(10, 10), sharex=True)
    
    # Row 1: Net Transport
    plot_panel(axes[0, 0], time, df_psw_flux['flux_sv'].values, COLOR_PSW, 
               'PSW Net Transport (Sv)', 
               f'a) Polar Surface Water Net Transport — Fram Strait',
               show_xlabel=False)
    
    plot_panel(axes[0, 1], time, df_aw_flux['flux_sv'].values, COLOR_AW, 
               'AW Net Transport (Sv)', 
               f'b) Atlantic Water Net Transport — Eastern Boundary',
               show_xlabel=False)
    
    # Row 2: Temperature
    plot_panel(axes[1, 0], time, df_psw_props['temp'].values, COLOR_PSW, 
               'Potential Temperature (°C)', 
               'c) PSW Potential Temperature',
               show_xlabel=False)
    
    plot_panel(axes[1, 1], time, df_aw_props['temp'].values, COLOR_AW, 
               'Potential Temperature (°C)', 
               'd) AW Potential Temperature',
               show_xlabel=False)
    
    # Row 3: Salinity
    plot_panel(axes[2, 0], time, df_psw_props['sal'].values, COLOR_PSW, 
               'Salinity (psu)', 
               'e) PSW Salinity',
               show_xlabel=True)
    
    plot_panel(axes[2, 1], time, df_aw_props['sal'].values, COLOR_AW, 
               'Salinity (psu)', 
               'f) AW Salinity',
               show_xlabel=True)
    
    plt.tight_layout()

    for ax in axes[2, :]:
        ax.set_xticks([np.datetime64(f'{y}-01-01') for y in range(1995, 2026, 5)])
        ax.xaxis.set_major_formatter(dates.DateFormatter('%Y'))
        ax.tick_params(axis='x', labelsize=11)
    
    # Save figure
    figure_file = OUTPUT_DIR / 'figure_04_psw_aw_comparison.png'
    fig.savefig(figure_file, dpi=600, bbox_inches='tight')
    log_output_file(logger, figure_file, "6-panel comparison figure")
    plt.close(fig)

    # ========================================================================
    # WELCH'S T-TEST: AW TRANSPORT PERIOD COMPARISON
    # ========================================================================

    aw_transport = df_aw_flux['flux_sv'].values
    time_aw = pd.to_datetime(df_aw_flux['time'].values)

    period_1 = aw_transport[time_aw.year < SPLIT_YEAR]
    period_2 = aw_transport[time_aw.year >= SPLIT_YEAR]

    t_stat, p_value = scipy_stats.ttest_ind(period_1, period_2, equal_var=False)

    logger.info("Welch's t-test — AW transport period comparison:")
    logger.info(f"  Period 1 mean: {period_1.mean():.3f} Sv")
    logger.info(f"  Period 2 mean: {period_2.mean():.3f} Sv")
    logger.info(f"  Difference:    {period_2.mean() - period_1.mean():.3f} Sv")
    logger.info(f"  t-statistic:   {t_stat:.3f}")
    logger.info(f"  p-value:       {p_value:.4f}")

    # ========================================================================
    # WRITE METHODOLOGY DOCUMENTATION
    # ========================================================================
    
    methodology_file = write_methodology(OUTPUT_METHODS_DIR)
    
    log_completion(logger, start_time)
    
    logger.info("")
    logger.info("Output files:")
    logger.info(f"  Figure: {figure_file}")
    logger.info(f"  Methodology: {methodology_file}")
    logger.info(f"  Processed data: {PROCESSED_DIR}")
    logger.info("="*70)

except Exception as e:
    log_error(logger, e)
    raise

INFO     | ======================================================================
INFO     | Starting analysis: figure_04_PSW_and_AW
INFO     | Log file: outputs/logs/figure_04_PSW_and_AW_20260623_112616.log
INFO     | Timestamp: 2026-06-23 11:26:16
INFO     | ======================================================================
INFO     | Loading configuration...
INFO     | Configuration loaded successfully
INFO     |   PSW definition: S < 34.5 psu, ρ < 1027.7 kg/m³
INFO     |   AW definition: lon > 5.0°E, depth < 550.0 m
INFO     |   Split year: 2015
INFO     |   Bootstrap iterations: 1000
INFO     |   Rolling mean: enabled (window=12 months, 6 months masked at each end)
INFO     | 
INFO     | ======================================================================
INFO     | STARTING PSW AND AW COMPARISON ANALYSIS
INFO     | ======================================================================
INFO     | Loading GLORYS12 data from: ../../glorys12_with_density/glorys12_*_Greenland_Se